# UP Contingent Planning Tutorial

<a target="_blank" href="https://colab.research.google.com/github/aiplan4eu/up-cpor/blob/master/notebooks/CPOR_and_SDR_running_examples.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Contingent planning under partial observability and sensing actions is an importat problem in automated planning.
This notebook provides examples on using the contingent planning package of the unified planning framework. The package supports offline planning, where a complete plan graph is constructed, and online planning, where the planner interacts with the environment during execution, receving observations and computing which action to perfrom next.

For information about contingent planning, and the algorithms used here can be found at:


*   Shlomi Maliah, Radimir Komarnitsky, Guy Shani: Computing Contingent Plan Graphs using Online Planning. JAAMAS 16(1): 1:1-1:30 (2021)
*   Ronen I. Brafman, Guy Shani: Replanning in Domains with Partial Information and Sensing Actions. J. Artif. Intell. Res. 45: 565-600 (2012)

For questions or comments please contact Guy Shani - shanigu@bgu.ac.il.


# If you would like that the solution will be print change the parameter to True

In [ ]:
SOLUTION_PRINTED = False

def print_CPOR_sol(p_planNode, tab = ""):
    if p_planNode is not None:
        x = p_planNode
        print(tab, x.action_instance)
        for c in x.children:
            print_CPOR_sol(c[1], tab+"\t")

### Install

 We begin by installing the latest version of the up-cpor package.

In [ ]:
!pip install git+https://github.com/guyazran/up-cpor

# Loading Problems

We are now done with installations, and can start defining a problem that the planner can tackle. In this tutorial we demonstrate how problems can be loaded from pddl, but one can define a contingent problem through other methods, using the UP API.

In [ ]:
from unified_planning.io import PDDLReader

# Creating a PDDL reader
reader = PDDLReader()

prob_arr = ['blocks2', 'doors5', 'wumpus05']

# Offline Planning Example

We now deonstrate how to compute a complete plan graph for a contingent problem, where nodes are labeled by actions, and edges are labeled by observations. The package currently implements only the CPOR offline planner. We initialize the planner, and then call the solve method to compute a solution.

After a solution plan tree is computed, we can save the resulting plan to a file.

In [ ]:
import os

import unified_planning.environment as environment
from unified_planning.shortcuts import OneshotPlanner
from unified_planning.engines.results import PlanGenerationResultStatus

TESTS__URL = os.path.join("..", "tests")

for prob in prob_arr:
    print(f"###########################Problem: {prob} start###########################")
    # Parsing a PDDL problem from file
    problem = reader.parse_problem(
        os.path.join(TESTS_DIR, prob, 'd.pddl'),
        os.path.join(TESTS_DIR, prob, 'p.pddl')
    )

    env = environment.get_environment()
    env.factory.add_engine('CPORPlanning', 'up_cpor.engine', 'CPORImpl')

    with OneshotPlanner(name='CPORPlanning') as planner:
        result = planner.solve(problem)
        if SOLUTION_PRINTED:
            print_CPOR_sol(result.plan.root_node)
        if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
            print(f'{planner.name} found a valid plan!')
            print(f'Success')
        else:
            print('No plan found!')

# Using a UP Classical Planner Inside CPOR

CPOR (and SDR) operate by creating classical planning problems that model the partial knowledge, and solve them, to obtain a heuristic about which action to choose next. The CPOR package contains an internal impementation of the popular FF classical planner, by Joerg Hoffman. However, the package supports running any UP classical solver. We demonstrate here how the UP implementation of Tamer can be used instead of the internal FF.

In [ ]:
!pip install up-tamer

In [ ]:
import unified_planning.environment as environment
from unified_planning.engines.results import PlanGenerationResultStatus
from unified_planning.shortcuts import OneshotPlanner

TESTS_DIR = os.path.join("..", "tests")

for prob in prob_arr:
    print(f"###########################Problem: {prob} start###########################")
    # Parsing a PDDL problem from file
    problem = reader.parse_problem(
        os.path.join(TESTS_DIR, prob, 'd.pddl'),
        os.path.join(TESTS_DIR, prob, 'p.pddl')
    )

    env = environment.get_environment()
    env.factory.add_meta_engine('MetaCPORPlanning', 'up_cpor.engine', 'CPORMetaEngineImpl')

    with OneshotPlanner(name='MetaCPORPlanning[tamer]') as planner:
        result = planner.solve(problem)
        if SOLUTION_PRINTED:
            print_CPOR_sol(result.plan.root_node)
        if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
            print(f'{planner.name} found a valid plan!')
            print(f'Success')
        else:
            print('No plan found!')

#Online Contingent Planning

While in offline planning the planner computes a complete plan graph, in online planning we take a closed loop approach, where an agent interacts with the environment during execution.

The agent executes an action in the environment, and then receives an observation as a result in the action. In goal-based contingent planning this loop continues until the agent ensures that the goal has been achieved.

The CPOR package implements the SDR contingent (re)planner. SDR operates by translating the contingent problem into a classical problem, solving it using a classical solver, and then executing the resulting actions, if their preconditions hold. When an unexpected observation was received, SDR replans.

The code below demonstrates how SDR can be used, interacting with a simulated environment, which is also implemented inside the CPOR package. The while loop below implements the closed loop process.



In [ ]:
import unified_planning.environment as environment
from unified_planning.shortcuts import ActionSelector
from up_cpor.simulator import SDRSimulator

TESTS_DIR = os.path.join("..", "tests")

for prob in prob_arr:
    print(f"###########################Problem: {prob} start###########################")
    # Parsing a PDDL problem from file
    problem = reader.parse_problem(
        os.path.join(TESTS_DIR, prob, 'd.pddl'),
        os.path.join(TESTS_DIR, prob, 'p.pddl')
    )

    env = environment.get_environment()
    env.factory.add_engine('SDRPlanning', 'up_cpor.engine', 'SDRImpl')

    with ActionSelector(name='SDRPlanning', problem=problem) as solver:
        simulatedEnv = SDRSimulator(problem)
        while not simulatedEnv.is_goal_reached():
            action = solver.get_action()
            observation = simulatedEnv.apply(action)
            solver.update(observation)
            if SOLUTION_PRINTED:
              print(f"Action: {action}")
              if observation:
                print(f"Observation: {observation}")
        print(f'{solver.name} found a valid plan!')
        print(f'Success')